# Airbnb Paris – Cleaned
- Numerische/encodierte Features (Frequency-Encoding + OHE), Freitexte entfernt
- ICC-Label `is_top_rating` (rating == 5 → 1, rating <= 3 → 0); `row_id` als Join-Key
- Split 50/20/30 pro `SEED`; Imputation, Encoding und Scaler werden **nur auf Train** gefittet

In [ ]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = int(os.environ.get("SEED", 1))
SCRAPE_DATE = pd.Timestamp("2025-09-01")  # fixed reference; a global max would leak
print("SEED", SEED)

## Rohdaten laden
- `row_id` = `id` (stabiler Join-Key für `cleaned_text` und die Splits)

In [ ]:
df = pd.read_csv("../../data/raw/airbnb_paris.csv", low_memory=False)
df["row_id"] = df["id"].astype("int64")
print("Rohdaten-Shape:", df.shape)

## Nicht-relevante / leere Spalten entfernen
- IDs/URLs/Scrape-Meta, Target-Sub-Scores (Leakage), **alle** Review-Counts/-Daten, Kalender-Meta, Duplikate, 100%-NaN

In [ ]:
columns_to_drop = [
    "id", "listing_url", "scrape_id", "last_scraped", "source", "picture_url",
    "host_id", "host_url", "host_name", "host_thumbnail_url", "host_picture_url", "license",
    "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin",
    "review_scores_communication", "review_scores_location", "review_scores_value",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d", "number_of_reviews_ly",
    "first_review", "last_review", "reviews_per_month",
    "calendar_updated", "calendar_last_scraped", "bathrooms_text",
    "host_listings_count", "host_total_listings_count",
    "minimum_minimum_nights", "maximum_minimum_nights",
    "minimum_maximum_nights", "maximum_maximum_nights",
    "minimum_nights_avg_ntm", "maximum_nights_avg_ntm",
    "has_availability", "host_neighbourhood", "neighbourhood",
    "price", "estimated_revenue_l365d", "neighbourhood_group_cleansed",
]
df = df.drop(columns=[c for c in columns_to_drop if c in df.columns])
print("Nach Drop:", df.shape)

## ICC-Target erstellen
- Inlier (`is_top_rating == 1`): rating == 5; Outlier (0): rating <= 3; Mittelfeld und NaN verworfen

In [ ]:
df = df.dropna(subset=["review_scores_rating"]).copy()
df = df.loc[(df["review_scores_rating"] == 5.0) | (df["review_scores_rating"] <= 3.0)].copy()
df["is_top_rating"] = (df["review_scores_rating"] == 5.0).astype(int)
print("Nach Target:", df.shape, "| Outlier-Rate:", round((df["is_top_rating"] == 0).mean(), 4))

## Feature Engineering
- host_tenure_days gegen ein **festes Scrape-Datum**, Prozent-Raten → float, Listen-Counts, host_location-Flags, Booleans t/f → 0/1
- Rein zeilenweise, damit seed-unabhängig

In [ ]:
df["host_since"] = pd.to_datetime(df["host_since"], errors="coerce")
df["host_tenure_days"] = (SCRAPE_DATE - df["host_since"]).dt.days
df = df.drop(columns=["host_since"])

df["host_response_rate"] = pd.to_numeric(df["host_response_rate"].astype(str).str.rstrip("%"), errors="coerce")
df["host_acceptance_rate"] = pd.to_numeric(df["host_acceptance_rate"].astype(str).str.rstrip("%"), errors="coerce")

df["amenities_count"] = df["amenities"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
df["host_verifications_count"] = df["host_verifications"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
df = df.drop(columns=["amenities", "host_verifications"])

loc = df["host_location"].fillna("")
df["host_in_paris"] = loc.str.contains("Paris", case=False, na=False).astype(int)
df["host_in_france"] = loc.str.contains("France", case=False, na=False).astype(int)
df["host_location_missing"] = df["host_location"].isna().astype(int)
df = df.drop(columns=["host_location"])

for c in ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "instant_bookable"]:
    df[c] = df[c].map({"t": 1, "f": 0})
print("Nach FE:", df.shape)

## Split 50/20/30
- Stratifiziert über das Label, Schlüssel `row_id`
- Zeilen-Universum (seed-unabhängig) und Split-Zuordnung (pro Seed) werden separat abgelegt

In [ ]:
rows = pd.DataFrame({"row_id": df["row_id"].values, "is_top_rating": df["is_top_rating"].values})
tr_id, rest_id = train_test_split(rows["row_id"], train_size=0.5,
                                  stratify=rows["is_top_rating"], random_state=SEED)
rest = rows[rows["row_id"].isin(rest_id)]
val_id, te_id = train_test_split(rest["row_id"], train_size=0.4,
                                 stratify=rest["is_top_rating"], random_state=SEED)

split = rows[["row_id"]].copy()
split["split"] = np.where(split["row_id"].isin(tr_id), "train",
                          np.where(split["row_id"].isin(val_id), "val", "test"))
tr = df["row_id"].isin(tr_id).values

os.makedirs("../../data/splits", exist_ok=True)
rows.to_csv("../../data/splits/rows_airbnb_paris.csv", index=False)
split.to_csv(f"../../data/splits/split_airbnb_paris_seed{SEED}.csv", index=False)
print(split["split"].value_counts().to_dict())
print("Outlier-Rate je Split:", (1 - rows.groupby(split["split"])["is_top_rating"].mean()).round(4).to_dict())

## Imputation & Konstanten-Drop – nur auf Train gefittet
- Numerisch: Median; Boolean: Modus; Kategorisch: "unknown"
- Konstante numerische Spalten werden **anhand von Train** bestimmt (entfernt u. a. die 100%-NaN-Reste)

In [ ]:
text_cols = ["name", "description", "neighborhood_overview", "host_about"]
categorical_cols = ["host_response_time", "neighbourhood_cleansed", "property_type", "room_type"]
bool_cols = ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "instant_bookable",
             "host_in_paris", "host_in_france", "host_location_missing"]
non_numeric = set(text_cols + categorical_cols + bool_cols + ["is_top_rating", "review_scores_rating", "row_id"])

for c in [c for c in df.columns if c not in non_numeric]:
    df[c] = df[c].fillna(df.loc[tr, c].median())
for c in bool_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df.loc[tr, c].mode().iloc[0])
for c in categorical_cols:
    df[c] = df[c].fillna("unknown")

constant_cols = [c for c in df.columns
                 if c not in text_cols and pd.api.types.is_numeric_dtype(df[c])
                 and df.loc[tr, c].nunique(dropna=False) <= 1]
df = df.drop(columns=constant_cols)
print("Konstante gedroppt:", constant_cols)

## Encoding & Spaltennamen
- Frequency-Encoding (neighbourhood_cleansed, property_type) aus Train; ungesehene Kategorien → 0
- OHE: host_response_time, room_type; danach Spaltennamen auf snake_case normalisieren

In [ ]:
ohe_cols = ["host_response_time", "room_type"]
freq_cols = ["neighbourhood_cleansed", "property_type"]

# frequency encoding fitted on train; categories unseen in train -> 0
for c in freq_cols:
    df[c + "_freq"] = df[c].map(df.loc[tr, c].value_counts(normalize=True)).fillna(0.0)
df = df.drop(columns=freq_cols)
df = pd.get_dummies(df, columns=ohe_cols, dummy_na=False, dtype=int)

df.columns = [re.sub(r"_+", "_", re.sub(r"[^a-zA-Z0-9]+", "_",
              unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode("ascii")).lower()).strip("_")
              for c in df.columns]

## Skalierung – nur auf Train gefittet
- StandardScaler auf numerische + freq-Spalten (Boolean, OHE, Target, row_id ausgenommen)

In [ ]:
exclude = set(text_cols + bool_cols + ["is_top_rating", "review_scores_rating", "row_id"])
exclude.update([c for c in df.columns if any(c.startswith(p + "_") for p in ohe_cols)])
scale_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
scaler = StandardScaler().fit(df.loc[tr, scale_cols])
df[scale_cols] = scaler.transform(df[scale_cols])
print("Skaliert:", len(scale_cols), "Spalten")

## Speichern
- `review_scores_rating` und Freitexte entfernen; `row_id` bleibt

In [ ]:
out = df.drop(columns=["review_scores_rating"] + text_cols).reset_index(drop=True)
out.to_csv(f"../../data/preprocessed/cleaned_airbnb_paris_seed{SEED}.csv", index=False)
print("Shape:", out.shape)

## Verifikation
- Gegenprobe zum Leakage: Scaler-Mittelwert auf Train ≈ 0, auf Test ≠ 0

In [ ]:
assert out.isna().sum().sum() == 0
assert out["row_id"].is_unique
print("Mittelwert Train:", round(float(out.loc[tr, scale_cols].to_numpy().mean()), 6))
print("Mittelwert Test :", round(float(out.loc[(split["split"] == "test").values, scale_cols].to_numpy().mean()), 6))